In [11]:
# ============================================================
# FAST LIVE DATA PIPELINE — SETUP
# ============================================================

import ee
import pandas as pd
import numpy as np
import os

from datetime import datetime, timedelta

ee.Initialize()

print("✅ GEE initialized")

✅ GEE initialized


In [12]:
# ============================================================
# JOSHIMATH STUDY AREA
# ============================================================

JOSHIMATH_LON = 79.566
JOSHIMATH_LAT = 30.556

joshimath_point = ee.Geometry.Point(
    [JOSHIMATH_LON, JOSHIMATH_LAT]
)

study_area = joshimath_point.buffer(10000)

print("✅ Joshimath 10 km study area ready")

✅ Joshimath 10 km study area ready


In [17]:
# ============================================================
# FAST DAILY GEE DATA EXTRACTION — SAFE VERSION
# Handles missing SMAP / Sentinel-1 observations
# ============================================================

def get_fast_daily_data(date_string):

    print("Processing:", date_string)

    # --------------------------------------------------------
    # DATE RANGE
    # --------------------------------------------------------

    start = ee.Date(date_string)
    end = start.advance(1, "day")


    # ========================================================
    # 1. GPM — RAINFALL
    # ========================================================

    gpm = (
        ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
        .filterDate(start, end)
        .filterBounds(study_area)
    )

    gpm_count = gpm.size()

    # Safe rainfall calculation
    rainfall = ee.Algorithms.If(
        gpm_count.gt(0),

        gpm.mean()
        .select("precipitation")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=10000,
            maxPixels=1e7
        )
        .get("precipitation"),

        None
    )


    # ========================================================
    # 2. SMAP — SOIL MOISTURE
    # ========================================================

    smap = (
        ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
        .filterDate(start, end)
        .filterBounds(study_area)
    )

    smap_count = smap.size()

    # IMPORTANT:
    # Only select soil_moisture_am if an image exists

    soil_moisture = ee.Algorithms.If(
        smap_count.gt(0),

        smap.mean()
        .select("soil_moisture_am")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=10000,
            maxPixels=1e7
        )
        .get("soil_moisture_am"),

        None
    )


    # ========================================================
    # 3. SENTINEL-1
    # ========================================================

    s1 = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(start, end)
        .filterBounds(study_area)
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
    )

    s1_count = s1.size()


    # --------------------------------------------------------
    # Sentinel-1 VV
    # --------------------------------------------------------

    sentinel_vv = ee.Algorithms.If(
        s1_count.gt(0),

        s1.mean()
        .select("VV")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("VV"),

        None
    )


    # --------------------------------------------------------
    # Sentinel-1 VH
    # --------------------------------------------------------

    sentinel_vh = ee.Algorithms.If(
        s1_count.gt(0),

        s1.mean()
        .select("VH")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("VH"),

        None
    )


    # ========================================================
    # 4. ELEVATION
    # ========================================================

    elevation_image = (
        ee.Image("USGS/SRTMGL1_003")
        .select("elevation")
    )

    elevation = (
        elevation_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("elevation")
    )


    # ========================================================
    # 5. SLOPE
    # ========================================================

    slope_image = ee.Terrain.slope(
        elevation_image
    )

    slope = (
        slope_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("slope")
    )


    # ========================================================
    # 6. ONE GEE REQUEST
    # ========================================================

    result = ee.Dictionary({

        "date": date_string,

        "rainfall": rainfall,

        "soil_moisture": soil_moisture,

        "sentinel1_vv": sentinel_vv,

        "sentinel1_vh": sentinel_vh,

        "elevation": elevation,

        "slope": slope,

        "gpm_count": gpm_count,

        "smap_count": smap_count,

        "sentinel1_count": s1_count

    }).getInfo()


    # ========================================================
    # 7. PRINT DATA AVAILABILITY
    # ========================================================

    print()
    print("GPM observations      :", result["gpm_count"])
    print("SMAP observations     :", result["smap_count"])
    print("Sentinel-1 observations:", result["sentinel1_count"])

    print()
    print("✅ Daily GEE extraction completed")

    return result

In [18]:
# ============================================================
# TEST
# ============================================================

test_date = (
    datetime.now() - timedelta(days=1)
).strftime("%Y-%m-%d")

fast_result = get_fast_daily_data(
    test_date
)

print()
print("======================================")
print("LIVE DAILY DATA")
print("======================================")

for key, value in fast_result.items():
    print(f"{key}: {value}")

Processing: 2026-09-05

GPM observations      : 0
SMAP observations     : 0
Sentinel-1 observations: 0

✅ Daily GEE extraction completed

LIVE DAILY DATA
date: 2026-09-05
elevation: 2731.5671510280154
gpm_count: 0
rainfall: None
sentinel1_count: 0
sentinel1_vh: None
sentinel1_vv: None
slope: 35.31415903321138
smap_count: 0
soil_moisture: None


In [19]:
# ============================================================
# CHECK RECENT GEE DATA AVAILABILITY
# ============================================================

from datetime import datetime, timedelta

end_date = datetime.now()
start_date = end_date - timedelta(days=7)

start_string = start_date.strftime("%Y-%m-%d")
end_string = end_date.strftime("%Y-%m-%d")

print("Checking:")
print(start_string, "to", end_string)
print()

# ------------------------------------------------------------
# GPM
# ------------------------------------------------------------

gpm = (
    ee.ImageCollection("NASA/GPM_L3/IMERG_V07")
    .filterDate(start_string, end_string)
    .filterBounds(study_area)
)

print("GPM images:", gpm.size().getInfo())


# ------------------------------------------------------------
# SMAP
# ------------------------------------------------------------

smap = (
    ee.ImageCollection("NASA/SMAP/SPL3SMP_E/005")
    .filterDate(start_string, end_string)
    .filterBounds(study_area)
)

print("SMAP images:", smap.size().getInfo())


# ------------------------------------------------------------
# Sentinel-1
# ------------------------------------------------------------

s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterDate(start_string, end_string)
    .filterBounds(study_area)
    .filter(
        ee.Filter.eq("instrumentMode", "IW")
    )
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation",
            "VV"
        )
    )
    .filter(
        ee.Filter.listContains(
            "transmitterReceiverPolarisation",
            "VH"
        )
    )
)

print("Sentinel-1 images:", s1.size().getInfo())

Checking:
2026-08-30 to 2026-09-06

GPM images: 284
SMAP images: 0
Sentinel-1 images: 2


In [20]:
# ============================================================
# SHOW AVAILABLE IMAGE DATES
# ============================================================

def show_dates(collection, name):

    dates = (
        collection
        .aggregate_array("system:time_start")
        .getInfo()
    )

    print()
    print("==========", name, "==========")

    if not dates:
        print("No observations found.")
        return

    for timestamp in dates:

        date = datetime.fromtimestamp(
            timestamp / 1000
        )

        print(
            date.strftime(
                "%Y-%m-%d %H:%M"
            )
        )


show_dates(gpm, "GPM")
show_dates(smap, "SMAP")
show_dates(s1, "SENTINEL-1")


========== GPM ==========
2026-08-30 05:30
2026-08-30 06:00
2026-08-30 06:30
2026-08-30 07:00
2026-08-30 07:30
2026-08-30 08:00
2026-08-30 08:30
2026-08-30 09:00
2026-08-30 09:30
2026-08-30 10:00
2026-08-30 10:30
2026-08-30 11:00
2026-08-30 11:30
2026-08-30 12:00
2026-08-30 12:30
2026-08-30 13:00
2026-08-30 13:30
2026-08-30 14:00
2026-08-30 14:30
2026-08-30 15:00
2026-08-30 15:30
2026-08-30 16:00
2026-08-30 16:30
2026-08-30 17:00
2026-08-30 17:30
2026-08-30 18:00
2026-08-30 18:30
2026-08-30 19:00
2026-08-30 19:30
2026-08-30 20:00
2026-08-30 20:30
2026-08-30 21:00
2026-08-30 21:30
2026-08-30 22:00
2026-08-30 22:30
2026-08-30 23:00
2026-08-30 23:30
2026-08-31 00:00
2026-08-31 00:30
2026-08-31 01:00
2026-08-31 01:30
2026-08-31 02:00
2026-08-31 02:30
2026-08-31 03:00
2026-08-31 03:30
2026-08-31 04:00
2026-08-31 04:30
2026-08-31 05:00
2026-08-31 05:30
2026-08-31 06:00
2026-08-31 06:30
2026-08-31 07:00
2026-08-31 07:30
2026-08-31 08:00
2026-08-31 08:30
2026-08-31 09:00
2026-08-31 09:30
2026